In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
# Lets start with loading and cleaning/preparing the dataframe for easier use

#raw dataset
file_path = 'ai4i2020.csv'
df = pd.read_csv(file_path)
#drop identifier columns
df_clean = df.drop(columns=['UDI','Product ID'])
#renaming columns for snake_case and cleaner names
rename_dict = {
    "Type": "type", "Air temperature [K]": "air_temperature",
    "Process temperature [K]": "process_temperature", "Rotational speed [rpm]": "rotational_speed",
    "Torque [Nm]": "torque", "Tool wear [min]": "tool_wear",
    "Machine failure": "machine_failure", "TWF": "twf", "HDF": "hdf", 
    "PWF": "pwf", "OSF": "osf", "RNF": "rnf"
}
df_clean = df_clean.rename(columns=rename_dict)

# mapping the strings of type to intergers
type_mapping = {"L": 0, "M": 1, "H": 2}
df_clean["type"] = df_clean["type"].map(type_mapping)
df_clean.head(2)


,type,air_temperature,process_temperature,rotational_speed,torque,tool_wear,machine_failure,twf,hdf,pwf,osf,rnf
0,1,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,0,298.2,308.7,1408,46.3,3,0,0,0,0,0,0


In [2]:
# extra features related to failure modes
df_clean["temperature_differences"] = df_clean["process_temperature"] - df_clean["air_temperature"]
df_clean["power_proxy"] = df_clean["torque"] * df_clean["rotational_speed"]
# Z-scoring scales for continous sensor features
features_to_scale = ["air_temperature","process_temperature","rotational_speed","torque","tool_wear",
                     "temperature_differences","power_proxy"]
scaler = StandardScaler()
df_scaled = df_clean.copy()
df_scaled[features_to_scale] = scaler.fit_transform(df_clean[features_to_scale])

#separate features (X) and target (Y)
columns_to_exclude = ["machine_failure","twf","hdf","pwf","osf","rnf"]
X = df_scaled.drop(columns = columns_to_exclude)

y = df_scaled["machine_failure"]

# stratify train_test split (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state =42, stratify=y)

print(f"Data prepared sucessfully! X_train_shape = {X_train.shape}")

Data prepared sucessfully! X_train_shape = (8000, 8)


In [3]:
# calculate the ratio between failures and normal so we can use the xgboost parameter scale_pos_weight later
imbalance_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(imbalance_ratio)     
# about 28.5 , that is, for every broken machine , more than 28 work normally

28.52029520295203


In [4]:
# now we try XGboost
from xgboost import XGBClassifier
#initialize the model
xgb_model = XGBClassifier(scale_pos_weight=imbalance_ratio, random_state = 42)
#train the model 
xgb_model.fit(X_train, y_train)
print("--- XGBOOST BASIC TRAINING COMPLETED ---")

--- XGBOOST BASIC TRAINING COMPLETED ---


In [5]:
# predictions 
from sklearn.metrics import classification_report, confusion_matrix
y_pred_xgb = xgb_model.predict(X_test)
# display results
print('--- XGBOOST PERFORMANCE REPORT ---')
print(classification_report(y_test, y_pred_xgb))

--- XGBOOST PERFORMANCE REPORT ---
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.85      0.81      0.83        68

    accuracy                           0.99      2000
   macro avg       0.92      0.90      0.91      2000
weighted avg       0.99      0.99      0.99      2000



In [11]:
# GridSearchCV do check different number of leaves
from sklearn.model_selection import GridSearchCV
# 1) Define the grid of parameters we want to test
# max_depth: controls how deep each tree can grow
# n_estimators: number of trees in the forest
param_grid = {"max_depth": [3,5,7], "n_estimators": [50,100,150]}
print('Parameter grid defined successfully! ')
print(param_grid)

Parameter grid defined successfully! 
{'max_depth': [3, 5, 7], 'n_estimators': [50, 100, 150]}


In [12]:
# 1) initialize gridCV engine
grid_search = GridSearchCV(estimator = xgb_model, param_grid = param_grid, scoring = "f1",
                          cv = 3, verbose = 1, n_jobs = 1)
# 2) Run the exhaustive automated search on our training data
grid_search.fit(X_train, y_train)

# 3) Print the best parameters found by the engine

print("\n--- BEST PARAMETERS FOUND--- ")
print(grid_search.best_params_)

Fitting 3 folds for each of 9 candidates, totalling 27 fits

--- BEST PARAMETERS FOUND--- 
{'max_depth': 5, 'n_estimators': 150}


In [15]:
# 1) initialise the model with the best parameters found before
best_xgb = grid_search.best_estimator_
# 2) do the predctions with it
y_pred_best = best_xgb.predict(X_test)
print("    --- FINE TUNED XGBOOST PERFORMANCE REPORT ---")
print(classification_report(y_test,y_pred_best))


    --- FINE TUNED XGBOOST PERFORMANCE REPORT ---
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.79      0.79      0.79        68

    accuracy                           0.99      2000
   macro avg       0.89      0.89      0.89      2000
weighted avg       0.99      0.99      0.99      2000



In [16]:
# Print the final confusion matrix to see the exact count of hits and misses
print("--- FINAL CONFUSION MATRIX ---")
print(confusion_matrix(y_test, y_pred_best))


--- FINAL CONFUSION MATRIX ---
[[1918   14]
 [  14   54]]
